# Lista 02 — Geometria Computacional

Notebook de resolução incremental da Lista 2.

## Contexto e convenções

Consultar `docs/geometry_conventions.md` antes de registrar cada solução.

### Ambiente e visualização

In [85]:
# imports, dirs, definitions, and constants
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists()
)

ARTIFACTS_DIR = (
    ROOT
    / "assignments"
    / "02-geometry"
    / "artifacts"
)

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)


FIGURE_WIDTH = 900
FIGURE_HEIGHT = 650


def close_path(vertices):
    vertices = np.asarray(vertices, dtype=np.float64)
    return np.vstack((vertices, vertices[0]))


def finalize_figure(
    fig,
    filename,
    *,
    width=FIGURE_WIDTH,
    height=FIGURE_HEIGHT,
    equal_axes=True,
    showlegend=True,
):
    fig.update_layout(
        width=width,
        height=height,
        showlegend=showlegend,
    )

    if equal_axes:
        fig.update_xaxes(
            zeroline=True,
            scaleanchor="y",
            scaleratio=1,
        )
        fig.update_yaxes(zeroline=True)

    fig.write_html(
        ARTIFACTS_DIR / filename,
        include_plotlyjs="cdn",
    )

    return fig

## Questão 1

### Implementação

A primitiva `pseudo_angle` associa a direção de um vetor não nulo a um
valor no intervalo $[0,8)$, preservando a ordem angular sem utilizar
funções trigonométricas.

O mapeamento percorre os quatro quadrantes no sentido anti-horário,
iniciando no semieixo positivo de $x$. Cada quadrante ocupa dois valores
da escala.

O vetor nulo não possui direção e, portanto, não possui pseudo-ângulo
definido.

Para um vetor não nulo, o cálculo do pseudo-ângulo utiliza, em cada execução:

- **3 comparações** para determinar o quadrante e o trecho correspondente;
- **1 divisão**;
- **no máximo 1 soma ou subtração** para ajustar o valor ao intervalo;
- **até 2 inversões de sinal**;
- **0 multiplicações**;
- **0 raízes quadradas**;
- **0 operações de valor absoluto**;
- **0 funções trigonométricas**.

A implementação inclui ainda uma verificação adicional para o vetor nulo por meio de `is_zero`. Essa validação **não é necessária para o cálculo do pseudo-ângulo solicitado na questão** e, portanto, não é considerada na contagem de operações acima. Ela foi acrescentada como uma decisão de projeto para tratar explicitamente uma entrada sem direção definida e para manter a mesma política numérica utilizada nas demais primitivas geométricas (usando a tolerância numérica comum do projeto).

Como `is_zero` utiliza uma tolerância numérica e `std::abs` internamente, essa etapa acrescenta até dois testes de nulidade e operações de valor absoluto. Esse custo corresponde exclusivamente à validação adicional da entrada, e não ao cálculo do pseudo-ângulo propriamente dito.

Assim, desconsiderando essa verificação de segurança, o pseudo-ângulo é obtido com um número constante e pequeno de operações, em particular com **uma única divisão** e sem o cálculo de norma, valor absoluto ou funções trigonométricas como $\arccos$.

In [41]:
import numpy as np
import tca._core as core

print("Testing pseudo_angle function:")
result = core.pseudo_angle(np.array([1.0, 0.0], dtype=np.float64))
print(f"pseudo_angle([1.0, 0.0]) = {result}")

result = core.pseudo_angle(np.array([0.0, 1.0], dtype=np.float64))
print(f"pseudo_angle([0.0, 1.0]) = {result}")

result = core.pseudo_angle(np.array([-1.0, 0.0], dtype=np.float64))
print(f"pseudo_angle([-1.0, 0.0]) = {result}")

result = core.pseudo_angle(np.array([0.0, -1.0], dtype=np.float64))
print(f"pseudo_angle([0.0, -1.0]) = {result}")

try:
    core.pseudo_angle(np.array([0.0, 0.0], dtype=np.float64))
except ValueError as error:
    print(f"Caught expected exception for zero vector: {error}")
else:
    print(f"pseudo_angle([0.0, 0.0]) = {result}")

try:
    core.pseudo_angle(np.array([1e-13,1e-13], dtype=np.float64))
except ValueError as error:
    print(f"Caught expected exception for almost zero ([1e-13,1e-13]) vector: {error}")
else:
    print(f"pseudo_angle([1e-13,1e-13]) = {result}")

Testing pseudo_angle function:
pseudo_angle([1.0, 0.0]) = 0.0
pseudo_angle([0.0, 1.0]) = 2.0
pseudo_angle([-1.0, 0.0]) = 4.0
pseudo_angle([0.0, -1.0]) = 6.0
Caught expected exception for zero vector: pseudo_angle: vector cannot be the zero vector
Caught expected exception for almost zero ([1e-13,1e-13]) vector: pseudo_angle: vector cannot be the zero vector


### Conjunto de pontos de teste

Foi construído o produto cartesiano

\[
\{-3,-2,-1,0,1,2,3\}
\times
\{-3,-2,-1,0,1,2,3\},
\]

excluindo-se o ponto \((0,0)\), uma vez que o vetor nulo não possui direção e, portanto, não possui pseudo-ângulo definido. O conjunto resultante contém 48 pontos.

Para testar o comportamento da implementação próximo às regiões de transição da função de pseudo-ângulo, são adicionadas pequenas perturbações independentes às coordenadas dos pontos localizados:

- nas diagonais, isto é, quando \(|x|=|y|\) e \(|x|>1\);
- sobre o eixo \(y\), quando \(x=0\) e \(|y|>1\);
- sobre o eixo \(x\), quando \(y=0\) e \(|x|>1\).

Cada perturbação é amostrada uniformemente no intervalo

\[
[-0.001; 0.001].
\]

Dessa forma, são criados pontos ligeiramente deslocados para ambos os lados das fronteiras utilizadas pelas comparações internas do algoritmo, permitindo avaliar seu comportamento próximo a eixos e diagonais.

As perturbações são produzidas por um gerador pseudoaleatório com *seed* fixa (`SEED = 42`), garantindo a reprodutibilidade do experimento. Por fim, a ordem dos 48 pontos também é aleatorizada utilizando o mesmo gerador.

In [42]:
import numpy as np

SEED = 42
SMALL_ERROR = 5e-2

rng = np.random.default_rng(SEED)

values = [-3, -2, -1, 0, 1, 2, 3]

points = np.array(
    [(x, y) for x in values for y in values if not (x == 0 and y == 0)],
    dtype=np.float64,
)

for point in points:
    x, y = point

    if abs(x) == abs(y) and abs(x) > 1:
        point[0] += rng.uniform(-SMALL_ERROR, SMALL_ERROR)
        point[1] += rng.uniform(-SMALL_ERROR, SMALL_ERROR)

    if x == 0 and abs(y) > 1:
        point[0] += rng.uniform(-SMALL_ERROR, SMALL_ERROR)
        point[1] += rng.uniform(-SMALL_ERROR, SMALL_ERROR)

    if y == 0 and abs(x) > 1:
        point[0] += rng.uniform(-SMALL_ERROR, SMALL_ERROR)
        point[1] += rng.uniform(-SMALL_ERROR, SMALL_ERROR)

rng.shuffle(points)

In [43]:
# função auxiliar de plotagem
import numpy as np
import plotly.graph_objects as go


def plot_point_order(
    points,
    order,
    *,
    annotations=None,
    values=None,
    value_name="value",
    title=None,
):
    points = np.asarray(points, dtype=np.float64)
    order = np.asarray(order)

    if annotations is None:
        labels = [str(value) for value in order]
    else:
        labels = [str(value) for value in annotations]

    customdata_columns = [order]

    if values is not None:
        values = np.asarray(values)
        customdata_columns.append(values)

    customdata = np.column_stack(customdata_columns)

    hovertemplate = (
        "x=%{x:.6f}<br>"
        "y=%{y:.6f}<br>"
        "order=%{customdata[0]:.0f}<br>"
    )

    if values is not None:
        hovertemplate += f"{value_name}=%{{customdata[1]:.6f}}<br>"

    hovertemplate += "<extra></extra>"

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=points[:, 0],
            y=points[:, 1],
            mode="markers+text",
            text=labels,
            textposition="top center",
            marker=dict(
                size=10,
                color=order,
                colorscale="Turbo",
                showscale=True,
                colorbar=dict(title="Order"),
            ),
            customdata=customdata,
            hovertemplate=hovertemplate,
        )
    )

    fig.update_layout(
        title=title,
        width=500,
        height=500,
        xaxis_title="x",
        yaxis_title="y",
        xaxis=dict(
            zeroline=True,
            scaleanchor="y",
            scaleratio=1,
        ),
        yaxis=dict(
            zeroline=True,
        ),
        showlegend=False,
    )

    return fig

In [44]:
import plotly.graph_objects as go

order = np.arange(1, len(points) + 1)

plot_point_order(
    points,
    order,
    title="Organização dos pontos (aleatóriamente ordenados)",
).show()

In [45]:
import tca._core as core

pseudo_angles = np.array(
    [core.pseudo_angle(point) for point in points]
)

pseudo_angles

array([7.50000000e+00, 6.00484766e+00, 1.50000000e+00, 3.00000000e+00,
       1.66666667e+00, 6.66666667e-01, 3.50000000e+00, 6.33333333e+00,
       2.50000000e+00, 9.71213446e-01, 4.33333333e+00, 1.99720991e+00,
       2.00000000e+00, 2.33333333e+00, 2.99770965e+00, 4.99873728e+00,
       5.66666667e+00, 5.50000000e+00, 3.66666667e+00, 4.50000000e+00,
       2.66666667e+00, 1.92040897e-02, 6.50000000e+00, 3.99334147e+00,
       5.33333333e+00, 6.00000000e+00, 7.33333333e+00, 1.98912668e+00,
       0.00000000e+00, 7.00555895e+00, 5.00000000e-01, 5.01114654e+00,
       5.99720878e+00, 3.33333333e-01, 4.00243542e+00, 7.00000000e+00,
       7.66666667e+00, 6.17276476e-03, 4.00000000e+00, 2.98543269e+00,
       1.33333333e+00, 5.00000000e+00, 3.33333333e+00, 1.00000000e+00,
       4.66666667e+00, 1.00731099e+00, 6.98392855e+00, 6.66666667e+00])

In [46]:
import plotly.graph_objects as go

point_order = np.arange(1, len(points) + 1)

labels = [f"{value:.3f}" for value in pseudo_angles]

plot_point_order(
    points,
    point_order,
    annotations=labels,
    values=pseudo_angles,
    value_name="pseudo_angle",
    title="Pseudo ângulos (não ordenados)",
).show()

In [47]:
import numpy as np

import tca._core as core
from tca.algorithms.sorting import sort


# Compute pseudo-angles in the original random point order.
pseudo_angles = np.array(
    [core.pseudo_angle(point) for point in points],
    dtype=np.float64,
)

# Sort a copy in-place using the C++ Merge Sort.
sorted_pseudo_angles = pseudo_angles.copy()
sort(sorted_pseudo_angles, method="merge", backend="cpp")

# Position of each original pseudo-angle in the sorted sequence.
orders = np.searchsorted(
    sorted_pseudo_angles,
    pseudo_angles,
) + 1

orders

array([47, 39, 11, 21, 12,  6, 23, 40, 17,  7, 28, 14, 15, 16, 20, 31, 36,
       35, 24, 29, 18,  3, 41, 25, 34, 38, 46, 13,  1, 45,  5, 33, 37,  4,
       27, 44, 48,  2, 26, 19, 10, 32, 22,  8, 30,  9, 43, 42])

In [48]:
import numpy as np
import plotly.graph_objects as go

plot_point_order(
    points,
    orders,
    annotations=labels,
    values=pseudo_angles,
    value_name="pseudo-angle",
    title="Pseudo ângulos (ordenados)",
).show()

## Questão 2

### Implementação

A função `entre(u, v, w)` utiliza o sinal do produto vetorial para
determinar a posição angular de $w$ em relação ao ângulo convexo
determinado por $u$ e $v$.

A convenção adotada é:

- `0`: $u$ e $v$ são colineares;
- `1`: $w$ pertence ao ângulo convexo, incluindo sua fronteira;
- `2`: $w$ está fora desse ângulo.

A decisão depende apenas de produtos vetoriais e comparações de sinais.

In [49]:
import numpy as np

from tca.geometry import entre

u = np.array([1.0, 0.0])
v = np.array([0.0, 1.0])

print(entre(u, v, np.array([1.0, 1.0])))    # 1
print(entre(u, v, np.array([-1.0, -1.0])))  # 2
print(entre(u, v, u))                       # 1
print(entre(u, v, v))                       # 1

1
2
1
1


In [50]:
u = np.array([1.0, 0.0])
v = np.array([2.0, 0.0])
w = np.array([0.0, 1.0])

print(entre(u, v, w))  # 0

0


In [51]:
u = np.array([1.0, 0.0])
v = np.array([0.0, 1.0])

# Compute ENTRE in the original random point order.
entre_points = np.array(
    [entre(u, v, point) for point in points],
    dtype=np.int64,
)

plot_point_order(
    points,
    orders,
    annotations=entre_points,
    values=entre_points,
    value_name="entre",
    title="Função ENTRE",
).show()

In [52]:
u = np.array([0.0, -1.0])
v = np.array([-1.0, 0.0])

# Compute ENTRE in the original random point order.
entre_points = np.array(
    [entre(u, v, point) for point in points],
    dtype=np.int64,
)

plot_point_order(
    points,
    orders,
    annotations=entre_points,
    values=entre_points,
    value_name="entre",
    title="Função ENTRE",
).show()

In [53]:
u = np.array([3.0, 2.0])
v = np.array([-1.5, -1.5])

# Compute ENTRE in the original random point order.
entre_points = np.array(
    [entre(u, v, point) for point in points],
    dtype=np.int64,
)

plot_point_order(
    points,
    orders,
    annotations=entre_points,
    values=entre_points,
    value_name="entre",
    title="Função ENTRE",
).show()

## Questão 3

## Questão 3 — Localização de ponto em relação a um triângulo

A localização de um ponto \(P\) em relação a um triângulo
\(P_1P_2P_3\) será realizada utilizando coordenadas baricêntricas.

Um ponto pode ser escrito como

\[
P = \lambda_1 P_1 + \lambda_2 P_2 + \lambda_3 P_3,
\]

com

\[
\lambda_1 + \lambda_2 + \lambda_3 = 1.
\]

As coordenadas baricêntricas podem ser calculadas por meio de razões
entre áreas orientadas. O sinal dessas coordenadas permite identificar
a posição do ponto em relação ao triângulo:

- todas positivas: ponto interior;
- nenhuma negativa e pelo menos uma nula: ponto na fronteira;
- pelo menos uma negativa: ponto exterior.

Foram implementadas três funções relacionadas:

- `oriented_area`: calcula a área orientada de um polígono;
- `barycentric_coordinates`: calcula \((\lambda_1,\lambda_2,\lambda_3)\);
- `locate_point_in_triangle`: classifica o ponto como exterior, interior ou fronteira.

In [54]:
import numpy as np

triangle = np.array(
    [
        [-2.0, -1.0],
        [ 2.0, -1.0],
        [ 0.0,  2.0],
    ],
    dtype=np.float64,
)

p1, p2, p3 = triangle

### Área orientada

A área orientada preserva, além do módulo da área, a orientação dos
vértices do polígono.

Para vértices ordenados no sentido anti-horário a área é positiva,
enquanto a inversão da ordem produz uma área negativa de mesmo módulo.

In [55]:
import numpy as np

from tca.geometry import oriented_area

n_vertices = 13
radius = 2.0

polygon = np.array(
    [
        [
            radius * np.cos(2 * np.pi * i / n_vertices),
            radius * np.sin(2 * np.pi * i / n_vertices),
        ]
        for i in range(n_vertices)
    ],
    dtype=np.float64,
)

try:
    area_ccw = oriented_area(polygon)
    area_cw = oriented_area(polygon[::-1].copy())
except ValueError as error:
    print(f"Erro esperado para polígono com {n_vertices} vértices: {error}")
else:
    print(f"Área orientada CCW: {area_ccw:.2f}")
    print(f"Área orientada CW:  {area_cw:.2f}")

Área orientada CCW: 12.08
Área orientada CW:  -12.08


In [56]:
area_ccw = oriented_area(triangle)
area_cw = oriented_area(triangle[::-1].copy())

print(f"Área orientada CCW: {area_ccw:.2f}") # 6.0
print(f"Área orientada CW:  {area_cw:.2f}") # -6.0

Área orientada CCW: 6.00
Área orientada CW:  -6.00


In [57]:
from tca.geometry import barycentric_coordinates

p = np.array([0.0, 0.0])
barycentric = barycentric_coordinates(p, triangle)
print(f"Barycentric coordinates[({p[0]:.2f}, {p[1]:.2f})]: ({barycentric[0]:.3f}, {barycentric[1]:.3f}, {barycentric[2]:.3f})")

p = p1
barycentric = barycentric_coordinates(p, triangle)
print(f"Barycentric coordinates[({p[0]:.2f}, {p[1]:.2f})]: ({barycentric[0]:.3f}, {barycentric[1]:.3f}, {barycentric[2]:.3f})")

p = p2
barycentric = barycentric_coordinates(p, triangle)
print(f"Barycentric coordinates[({p[0]:.2f}, {p[1]:.2f})]: ({barycentric[0]:.3f}, {barycentric[1]:.3f}, {barycentric[2]:.3f})")

p = p3
barycentric = barycentric_coordinates(p, triangle)
print(f"Barycentric coordinates[({p[0]:.2f}, {p[1]:.2f})]: ({barycentric[0]:.3f}, {barycentric[1]:.3f}, {barycentric[2]:.3f})")

p = (p1+2.*p3)/3.
barycentric = barycentric_coordinates(p, triangle)
print(f"Barycentric coordinates[({p[0]:.2f}, {p[1]:.2f})]: ({barycentric[0]:.3f}, {barycentric[1]:.3f}, {barycentric[2]:.3f})")


Barycentric coordinates[(0.00, 0.00)]: (0.333, 0.333, 0.333)
Barycentric coordinates[(-2.00, -1.00)]: (1.000, 0.000, 0.000)
Barycentric coordinates[(2.00, -1.00)]: (0.000, 1.000, 0.000)
Barycentric coordinates[(0.00, 2.00)]: (0.000, 0.000, 1.000)
Barycentric coordinates[(-0.67, 1.00)]: (0.333, 0.000, 0.667)


In [58]:
barycentric = np.array(
    [barycentric_coordinates(point, triangle) for point in points],
    dtype=np.float64,
)

lambda_1 = barycentric[:, 0]
lambda_2 = barycentric[:, 1]
lambda_3 = barycentric[:, 2]

In [59]:
# plot auxiliar
import numpy as np
import plotly.graph_objects as go


def plot_triangle_values(
    points,
    triangle,
    values,
    *,
    annotations=None,
    value_name="value",
    title=None,
    size=500,
):
    points = np.asarray(points, dtype=np.float64)
    triangle = np.asarray(triangle, dtype=np.float64)
    values = np.asarray(values)

    if annotations is None:
        labels = [f"{value:.2f}" for value in values]
    else:
        labels = [str(value) for value in annotations]

    limit = np.max(np.abs(values))
    closed_triangle = np.vstack((triangle, triangle[0]))

    fig = go.Figure()

    fig.add_trace(
        go.Scatter(
            x=points[:, 0],
            y=points[:, 1],
            mode="markers+text",
            text=labels,
            textposition="top center",
            marker=dict(
                size=10,
                color=values,
                colorscale="RdBu",
                cmin=-limit,
                cmax=limit,
                colorbar=dict(title=value_name),
            ),
            customdata=values,
            hovertemplate=(
                "x=%{x:.6f}<br>"
                "y=%{y:.6f}<br>"
                f"{value_name}=%{{customdata:.6f}}"
                "<extra></extra>"
            ),
        )
    )

    fig.add_trace(
        go.Scatter(
            x=closed_triangle[:, 0],
            y=closed_triangle[:, 1],
            mode="lines+markers",
            line=dict(width=2),
            marker=dict(size=9),
            hoverinfo="skip",
        )
    )

    fig.update_layout(
        title=title,
        width=size,
        height=size,
        xaxis_title="x",
        yaxis_title="y",
        xaxis=dict(
            zeroline=True,
            scaleanchor="y",
            scaleratio=1,
        ),
        yaxis=dict(zeroline=True),
        showlegend=False,
    )

    return fig

In [60]:
figs = []

for i in range(3):
    fig = plot_triangle_values(
        points,
        triangle,
        barycentric[:, i],
        value_name=f"λ{i+1}",
        title=f"Coordenada baricêntrica — λ{i+1}",
        size=500,
    )
    figs.append(fig)

In [61]:
from copy import deepcopy

from plotly.subplots import make_subplots

for fig in figs:
    fig.update_layout(
        showlegend=False,
    )

fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=["λ1", "λ2", "λ3"],
    horizontal_spacing=0.05,
)

colorbar_x = [0.30, 0.65, 1.00]

for col, source_fig in enumerate(figs, start=1):
    for source_trace in source_fig.data:
        trace = deepcopy(source_trace)

        # Reposiciona a colorbar de cada coordenada.
        if hasattr(trace, "marker") and trace.marker.showscale:
            trace.marker.colorbar.x = colorbar_x[col - 1]

        fig.add_trace(trace, row=1, col=col)

    yaxis = "y" if col == 1 else f"y{col}"

    fig.update_xaxes(
        zeroline=True,
        scaleanchor=yaxis,
        scaleratio=1,
        row=1,
        col=col,
    )

    fig.update_yaxes(
        zeroline=True,
        row=1,
        col=col,
    )

fig.update_layout(
    title="Coordenadas baricêntricas",
    width=1200,
    height=500,
    showlegend=False,
)

fig.show()

### Localização dos pontos

A partir dos sinais das três coordenadas baricêntricas, cada ponto é
classificado em relação ao triângulo segundo:

- `0`: exterior;
- `1`: interior;
- `2`: fronteira.

A fronteira é identificada considerando a tolerância numérica definida
para o projeto. A figura a seguir aplica a classificação ao mesmo conjunto
de pontos utilizado nos experimentos anteriores.

In [62]:
from tca.geometry import locate_point_in_triangle

locations = np.array(
    [locate_point_in_triangle(point, triangle) for point in points],
    dtype=np.int64,
)

fig = plot_point_order(
    points,
    locations,
    annotations=locations,
    values=locations,
    value_name="locate_point_in_triangle",
    title="Função locate_point_in_triangle",
)

# Corrige o título da escala.
fig.data[0].marker.colorbar.title = "Classe"

# Fecha o triângulo para desenhar sua fronteira.
closed_triangle = np.vstack((triangle, triangle[0]))

fig.add_trace(
    go.Scatter(
        x=closed_triangle[:, 0],
        y=closed_triangle[:, 1],
        mode="lines+markers",
        line=dict(width=2),
        marker=dict(size=8),
        name="Triângulo",
        hoverinfo="skip",
    )
)

fig.show()

## Questão 4

## Questão 4 — Clipping de um segmento por um triângulo

Considere um triângulo de vértices $P_1$, $P_2$ e $P_3$, e um segmento de extremos $A$ e $B$.

Seja um ponto $P$ pertencente ao segmento $AB$, descrito pela equação paramétrica

$$
P(t) = tA + (1-t)B,
\qquad t \in [0,1].
$$

Sejam

$$
(\alpha_1,\alpha_2,\alpha_3)
$$

as coordenadas baricêntricas de $A$ em relação ao triângulo $P_1P_2P_3$, e

$$
(\beta_1,\beta_2,\beta_3)
$$

as coordenadas baricêntricas de $B$.

Denotem-se por

$$
(\theta_1,\theta_2,\theta_3)
$$

as coordenadas baricêntricas de $P(t)$.

### Coordenadas baricêntricas ao longo do segmento

Por definição,

$$
\alpha_1 =
\frac{S_{AP_2P_3}}{S_{P_1P_2P_3}},
\qquad
\beta_1 =
\frac{S_{BP_2P_3}}{S_{P_1P_2P_3}},
$$

onde $S_{XYZ}$ representa a área orientada do triângulo $XYZ$,

$$
S_{XYZ}
=
\frac{1}{2}
\left(
\overrightarrow{OX}\times\overrightarrow{OY}
+
\overrightarrow{OY}\times\overrightarrow{OZ}
+
\overrightarrow{OZ}\times\overrightarrow{OX}
\right).
$$

Para o ponto

$$
P(t)=tA+(1-t)B,
$$

temos

$$
S_{PP_2P_3}
=
\frac{1}{2}
\left(
\overrightarrow{OP}\times\overrightarrow{OP_2}
+
\overrightarrow{OP_2}\times\overrightarrow{OP_3}
+
\overrightarrow{OP_3}\times\overrightarrow{OP}
\right).
$$

Como

$$
\overrightarrow{OP}
=
t\overrightarrow{OA}
+
(1-t)\overrightarrow{OB},
$$

segue que

$$
\begin{aligned}
S_{PP_2P_3}
&=
\frac{1}{2}
\Big[
\left(
t\overrightarrow{OA}
+
(1-t)\overrightarrow{OB}
\right)
\times
\overrightarrow{OP_2}
\\
&\qquad
+
\overrightarrow{OP_2}\times\overrightarrow{OP_3}
+
\overrightarrow{OP_3}
\times
\left(
t\overrightarrow{OA}
+
(1-t)\overrightarrow{OB}
\right)
\Big].
\end{aligned}
$$

Podemos ainda escrever

$$
\overrightarrow{OP_2}\times\overrightarrow{OP_3}
=
t
\left(
\overrightarrow{OP_2}\times\overrightarrow{OP_3}
\right)
+
(1-t)
\left(
\overrightarrow{OP_2}\times\overrightarrow{OP_3}
\right).
$$

Agrupando os termos associados a $A$ e a $B$,

$$
S_{PP_2P_3}
=
tS_{AP_2P_3}
+
(1-t)S_{BP_2P_3}.
$$

Logo,

$$
\theta_1
=
\frac{S_{PP_2P_3}}{S_{P_1P_2P_3}}
=
t
\frac{S_{AP_2P_3}}{S_{P_1P_2P_3}}
+
(1-t)
\frac{S_{BP_2P_3}}{S_{P_1P_2P_3}},
$$

e, portanto,

$$
\boxed{
\theta_1
=
t\alpha_1
+
(1-t)\beta_1
}.
$$

Analogamente,

$$
\boxed{
\theta_i
=
t\alpha_i
+
(1-t)\beta_i,
\qquad i=1,2,3.
}
$$

Assim, cada coordenada baricêntrica varia linearmente entre seu valor em $B$, quando $t=0$, e seu valor em $A$, quando $t=1$.

---

### a) Condição para que o segmento $AB$ esteja completamente contido no triângulo

Um ponto pertence ao triângulo, incluindo sua fronteira, quando todas as suas coordenadas baricêntricas são não negativas.

Portanto, basta que os dois extremos $A$ e $B$ pertençam ao triângulo:

$$
\alpha_i \geq 0
\qquad\text{e}\qquad
\beta_i \geq 0,
\qquad i=1,2,3.
$$

De fato, para qualquer ponto $P(t)$ do segmento,

$$
\theta_i(t)
=
t\alpha_i+(1-t)\beta_i.
$$

Como $t\in[0,1]$, $\theta_i(t)$ é uma combinação convexa de $\alpha_i$ e $\beta_i$. Portanto, se ambos forem não negativos,

$$
\theta_i(t)\geq0
$$

para todo $t\in[0,1]$.

Assim,

$$
\boxed{
\alpha_i\geq0
\;\land\;
\beta_i\geq0
\quad
\forall i
\quad\Longrightarrow\quad
AB\subseteq P_1P_2P_3.
}
$$

Caso seja considerado apenas o interior estrito do triângulo, deve-se utilizar

$$
\alpha_i>0
\qquad\text{e}\qquad
\beta_i>0.
$$

---

### b) Condições para garantir que o segmento $AB$ seja completamente exterior ao triângulo

Um ponto é exterior ao triângulo quando pelo menos uma de suas coordenadas baricêntricas é negativa.

Uma condição suficiente para garantir que todo o segmento $AB$ seja exterior consiste em existir uma mesma coordenada $i$ negativa nos dois extremos:

$$
\boxed{
\exists i
\quad\text{tal que}\quad
\alpha_i<0
\;\land\;
\beta_i<0.
}
$$

Nesse caso,

$$
\theta_i(t)
=
t\alpha_i
+
(1-t)\beta_i.
$$

Como $\alpha_i$ e $\beta_i$ são ambos negativos e $t\in[0,1]$, temos

$$
\theta_i(t)<0
$$

para todo $t\in[0,1]$.

Consequentemente, todo ponto $P(t)$ do segmento é exterior ao triângulo.

Geometricamente, isso significa que $A$ e $B$ estão no mesmo semiplano exterior determinado pela reta que contém o lado oposto ao vértice $P_i$.

Um caso particular ocorre quando

$$
\alpha_i=\beta_i.
$$

Nesse caso,

$$
\theta_i(t)
=
t\alpha_i+(1-t)\alpha_i
=
\alpha_i,
$$

ou seja, a coordenada baricêntrica permanece constante em todo o segmento.

Para $A\neq B$, isso significa que $AB$ é paralelo ao lado oposto a $P_i$.

Podem ocorrer três situações:

- se $\alpha_i=\beta_i<0$, todo o segmento permanece no semiplano exterior;
- se $\alpha_i=\beta_i=0$, todo o segmento está sobre a reta que contém o lado correspondente;
- se $\alpha_i=\beta_i>0$, o segmento está sobre uma reta paralela localizada no semiplano correspondente ao interior.

A condição $\alpha_i<0$ e $\beta_i<0$ para algum mesmo $i$ é, portanto, um teste suficiente para descartar imediatamente o segmento.

Caso esse teste não seja satisfeito, ele não permite concluir sozinho que existe interseção, sendo necessário analisar as retas que contêm os lados do triângulo.

---

### c) Interseções de $AB$ com as retas que contêm os lados do triângulo

Cada lado do triângulo está associado à anulação de uma das coordenadas baricêntricas.

Temos:

$$
\theta_1=0
\quad\Longleftrightarrow\quad
P\in\operatorname{reta}(P_2P_3),
$$

$$
\theta_2=0
\quad\Longleftrightarrow\quad
P\in\operatorname{reta}(P_3P_1),
$$

e

$$
\theta_3=0
\quad\Longleftrightarrow\quad
P\in\operatorname{reta}(P_1P_2).
$$

Como

$$
\theta_i(t)
=
t\alpha_i
+
(1-t)\beta_i,
$$

o parâmetro correspondente à interseção com a reta associada à coordenada $i$ pode ser obtido impondo

$$
\theta_i(t)=0.
$$

Assim,

$$
t\alpha_i+(1-t)\beta_i=0,
$$

$$
t\alpha_i+\beta_i-t\beta_i=0,
$$

$$
t(\alpha_i-\beta_i)=-\beta_i.
$$

Portanto, para

$$
\alpha_i\neq\beta_i,
$$

temos

$$
\boxed{
t_i
=
\frac{\beta_i}{\beta_i-\alpha_i}.
}
$$

O ponto correspondente da reta $AB$ é então

$$
\boxed{
X_i
=
t_iA
+
(1-t_i)B.
}
$$

Assim,

$$
t_1
=
\frac{\beta_1}{\beta_1-\alpha_1}
$$

determina a interseção de $AB$ com a reta que contém o lado $P_2P_3$;

$$
t_2
=
\frac{\beta_2}{\beta_2-\alpha_2}
$$

determina a interseção com a reta que contém o lado $P_3P_1$;

e

$$
t_3
=
\frac{\beta_3}{\beta_3-\alpha_3}
$$

determina a interseção com a reta que contém o lado $P_1P_2$.

Para que a interseção pertença ao próprio segmento $AB$, é necessário que

$$
0\leq t_i\leq1.
$$

Além disso, para que $X_i$ pertença ao lado do triângulo, e não apenas à reta que o contém, as outras duas coordenadas baricêntricas de $X_i$ devem ser não negativas.

Por exemplo, para a reta $P_2P_3$, temos

$$
\theta_1(t_1)=0.
$$

Para que $X_1$ pertença efetivamente ao segmento $P_2P_3$, deve ocorrer também

$$
\theta_2(t_1)\geq0
\qquad\text{e}\qquad
\theta_3(t_1)\geq0.
$$

O mesmo raciocínio vale para os demais lados.

Finalmente, se

$$
\alpha_i=\beta_i,
$$

o denominador da expressão de $t_i$ é nulo. Nesse caso não existe uma interseção isolada:

- se $\alpha_i=\beta_i\neq0$, o segmento $AB$ é paralelo à reta correspondente e não a intercepta;
- se $\alpha_i=\beta_i=0$, todo o segmento $AB$ pertence à reta correspondente.

Dessa forma, as coordenadas baricêntricas permitem obter diretamente os possíveis pontos de entrada e saída do segmento no triângulo, sem a necessidade de resolver explicitamente sistemas de interseção entre retas.

In [63]:
import numpy as np

from tca.geometry import barycentric_coordinates


triangleQ4 = np.array(
    [
        [0.0, 0.0],
        [4.0, 0.0],
        [0.0, 4.0],
    ],
    dtype=np.float64,
)

p1, p2, p3 = triangleQ4

A = np.array([-1.0, 1.0], dtype=np.float64)
B = np.array([3.0, 3.0], dtype=np.float64)

alpha = np.array(
    barycentric_coordinates(A, triangleQ4),
    dtype=np.float64,
)

beta = np.array(
    barycentric_coordinates(B, triangleQ4),
    dtype=np.float64,
)

alfa1, alfa2, alfa3 = alpha
beta1, beta2, beta3 = beta

print(
    f"A ({A[0]:.2f}, {A[1]:.2f}): "
    f"({alfa1:.3f}, {alfa2:.3f}, {alfa3:.3f})"
)

print(
    f"B ({B[0]:.2f}, {B[1]:.2f}): "
    f"({beta1:.3f}, {beta2:.3f}, {beta3:.3f})"
)

theta1 = lambda t: t * alfa1 + (1.0 - t) * beta1
theta2 = lambda t: t * alfa2 + (1.0 - t) * beta2
theta3 = lambda t: t * alfa3 + (1.0 - t) * beta3

t = np.linspace(0.0, 1.0, 11)

theta_values = np.column_stack(
    (
        theta1(t),
        theta2(t),
        theta3(t),
    )
)

direct_values = np.array(
    [
        barycentric_coordinates(
            ti * A + (1.0 - ti) * B,
            triangleQ4,
        )
        for ti in t
    ],
    dtype=np.float64,
)

A (-1.00, 1.00): (1.000, -0.250, 0.250)
B (3.00, 3.00): (-0.500, 0.750, 0.750)


In [64]:
import numpy as np
import plotly.graph_objects as go


t_plot = np.linspace(0.0, 1.0, 201)

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=t_plot,
        y=theta1(t_plot),
        mode="lines",
        name="θ₁",
    )
)

fig.add_trace(
    go.Scatter(
        x=t_plot,
        y=theta2(t_plot),
        mode="lines",
        name="θ₂",
    )
)

fig.add_trace(
    go.Scatter(
        x=t_plot,
        y=theta3(t_plot),
        mode="lines",
        name="θ₃",
    )
)

fig.add_hline(
    y=0.0,
    line_dash="dash",
)

fig.update_layout(
    title="Coordenadas baricêntricas ao longo do segmento AB",
    width=800,
    height=500,
    xaxis_title="t",
    yaxis_title="θᵢ(t)",
)

fig.show()

In [65]:
import numpy as np

from tca.geometry import locate_point_in_triangle


t_test = np.array([0.0, 1.0 / 3.0, 0.5, 3.0 / 4.0, 1.0])

for ti in t_test:
    point = ti * A + (1.0 - ti) * B
    location = locate_point_in_triangle(point, triangleQ4)

    print(
        f"t={ti:.3f}  "
        f"P=({point[0]:.3f}, {point[1]:.3f})  "
        f"location={location}"
    )

t=0.000  P=(3.000, 3.000)  location=0
t=0.333  P=(1.667, 2.333)  location=2
t=0.500  P=(1.000, 2.000)  location=1
t=0.750  P=(0.000, 1.500)  location=2
t=1.000  P=(-1.000, 1.000)  location=0


In [66]:
import numpy as np
import plotly.graph_objects as go

from tca.geometry import locate_point_in_triangle


# Pontos de interseção obtidos pelas coordenadas baricêntricas.
t_intersections = []

for alpha_i, beta_i in zip(
    (alfa1, alfa2, alfa3),
    (beta1, beta2, beta3),
):
    if not np.isclose(alpha_i, beta_i):
        ti = beta_i / (beta_i - alpha_i)

        if 0.0 <= ti <= 1.0:
            t_intersections.append(ti)


# Amostragem regular + valores exatos das interseções.
t_values = np.unique(
    np.concatenate(
        (
            np.linspace(0.0, 1.0, 11),
            np.array(t_intersections),
        )
    )
)

# P(t) = tA + (1-t)B
segment_points = np.array(
    [ti * A + (1.0 - ti) * B for ti in t_values],
    dtype=np.float64,
)

locations = np.array(
    [
        locate_point_in_triangle(point, triangleQ4)
        for point in segment_points
    ],
    dtype=np.int64,
)


# Fecha o triângulo para desenhá-lo.
closed_triangle = np.vstack((triangleQ4, triangleQ4[0]))

fig = go.Figure()


# Triângulo.
fig.add_trace(
    go.Scatter(
        x=closed_triangle[:, 0],
        y=closed_triangle[:, 1],
        mode="lines+markers",
        name="Triângulo",
        line=dict(width=2),
        marker=dict(size=8),
        hoverinfo="skip",
    )
)


# Segmento AB.
fig.add_trace(
    go.Scatter(
        x=[A[0], B[0]],
        y=[A[1], B[1]],
        mode="lines",
        name="Segmento AB",
        line=dict(width=2, dash="dash"),
        hoverinfo="skip",
    )
)


# Classes retornadas por locate_point_in_triangle.
location_names = {
    0: "Exterior",
    1: "Interior",
    2: "Fronteira",
}

for location, name in location_names.items():
    mask = locations == location

    fig.add_trace(
        go.Scatter(
            x=segment_points[mask, 0],
            y=segment_points[mask, 1],
            mode="markers+text",
            text=[f"t={ti:.3f}" for ti in t_values[mask]],
            textposition="top center",
            marker=dict(size=11),
            name=f"{location} — {name}",
            customdata=t_values[mask],
            hovertemplate=(
                "t=%{customdata:.6f}<br>"
                "x=%{x:.6f}<br>"
                "y=%{y:.6f}<br>"
                f"location={location} ({name})"
                "<extra></extra>"
            ),
        )
    )


# Identificação explícita dos extremos.
fig.add_trace(
    go.Scatter(
        x=[A[0], B[0]],
        y=[A[1], B[1]],
        mode="text",
        text=["A (t=1)", "B (t=0)"],
        textposition="bottom center",
        showlegend=False,
        hoverinfo="skip",
    )
)


fig.update_layout(
    title="Clipping de AB pelo triângulo — classificação ao longo de t",
    width=800,
    height=700,
    xaxis_title="x",
    yaxis_title="y",
    xaxis=dict(
        zeroline=True,
        scaleanchor="y",
        scaleratio=1,
    ),
    yaxis=dict(zeroline=True),
)

fig.show()

----

### Exemplo numérico

Considere o triângulo

$$
P_1=(0,0),\qquad
P_2=(4,0),\qquad
P_3=(0,4),
$$

e o segmento de extremos

$$
A=(-1,1),
\qquad
B=(3,3).
$$

Para esse triângulo, as coordenadas baricêntricas de um ponto
$P=(x,y)$ são

$$
\lambda_1=1-\frac{x}{4}-\frac{y}{4},
\qquad
\lambda_2=\frac{x}{4},
\qquad
\lambda_3=\frac{y}{4}.
$$

Assim, para $A=(-1,1)$,

$$
(\alpha_1,\alpha_2,\alpha_3)
=
\left(
1,-\frac14,\frac14
\right),
$$

e para $B=(3,3)$,

$$
(\beta_1,\beta_2,\beta_3)
=
\left(
-\frac12,\frac34,\frac34
\right).
$$

Observe que os dois extremos são exteriores ao triângulo:

- em $A$, temos $\alpha_2<0$;
- em $B$, temos $\beta_1<0$.

Entretanto, não existe um mesmo índice $i$ para o qual

$$
\alpha_i<0
\qquad\text{e}\qquad
\beta_i<0.
$$

Portanto, o teste suficiente de exterioridade do item (b) não permite
descartar o segmento.

O ponto genérico de $AB$ é

$$
P(t)=tA+(1-t)B,
\qquad t\in[0,1].
$$

Suas coordenadas baricêntricas são

$$
\theta_i(t)=t\alpha_i+(1-t)\beta_i.
$$

Logo,

$$
\theta_1(t)
=
t(1)+(1-t)\left(-\frac12\right)
=
\frac32t-\frac12,
$$

$$
\theta_2(t)
=
t\left(-\frac14\right)
+
(1-t)\frac34
=
\frac34-t,
$$

e

$$
\theta_3(t)
=
t\frac14+(1-t)\frac34
=
\frac34-\frac12t.
$$

#### Interseção com a reta $P_2P_3$

Essa reta é caracterizada por

$$
\theta_1=0.
$$

Portanto,

$$
\frac32t-\frac12=0,
$$

de onde

$$
t_1=\frac13.
$$

O ponto de interseção é

$$
X_1
=
\frac13 A+\frac23 B,
$$

portanto

$$
X_1
=
\left(
\frac53,\frac73
\right).
$$

De fato,

$$
\frac53+\frac73=4,
$$

confirmando que $X_1$ pertence à reta $P_2P_3$.

#### Interseção com a reta $P_3P_1$

Essa reta é caracterizada por

$$
\theta_2=0.
$$

Assim,

$$
\frac34-t=0,
$$

e portanto

$$
t_2=\frac34.
$$

O ponto correspondente é

$$
X_2
=
\frac34 A+\frac14 B
=
\left(
0,\frac32
\right),
$$

que pertence à reta $P_3P_1$, dada por $x=0$.

#### Reta $P_1P_2$

Para a terceira coordenada,

$$
\theta_3(t)
=
\frac34-\frac12t.
$$

Impondo $\theta_3=0$,

$$
t_3=\frac32.
$$

Como

$$
t_3\notin[0,1],
$$

essa interseção ocorre na reta que contém $P_1P_2$, mas fora do
segmento $AB$.

Portanto, a porção de $AB$ contida no triângulo corresponde a

$$
\boxed{
\frac13\leq t\leq\frac34
}
$$

e possui extremos

$$
\boxed{
X_1=
\left(
\frac53,\frac73
\right)
}
$$

e

$$
\boxed{
X_2=
\left(
0,\frac32
\right).
}
$$

## Questão 5

### Critério de disjunção

Dois triângulos são considerados disjuntos quando não possuem qualquer
ponto em comum.

Portanto, contato em um vértice ou em uma aresta também implica que os
triângulos não são disjuntos.

O algoritmo verifica:

1. interseções entre os pares de arestas;
2. contenção de um triângulo no outro.

Se nenhuma dessas situações ocorrer, os triângulos são disjuntos.

In [67]:
import numpy as np

from tca.geometry import triangles_are_disjoint


triangle_1 = np.array(
    [
        [0.0, 0.0],
        [4.0, 0.0],
        [0.0, 4.0],
    ],
    dtype=np.float64,
)

triangle_2 = np.array(
    [
        [5.0, 5.0],
        [7.0, 5.0],
        [5.0, 7.0],
    ],
    dtype=np.float64,
)

triangles_are_disjoint(triangle_1, triangle_2)

True

In [68]:
triangle_3 = np.array(
    [
        [2.0, 1.0],
        [5.0, 1.0],
        [2.0, 4.0],
    ],
    dtype=np.float64,
)

triangles_are_disjoint(triangle_1, triangle_3) # sobreposicao

False

In [69]:
import numpy as np

triangles = {
    "T0_base": np.array(
        [
            [0.0, 0.0],
            [4.0, 0.0],
            [0.0, 4.0],
        ],
        dtype=np.float64,
    ),
    "T1_disjoint": np.array(
        [
            [5.0, 5.0],
            [7.0, 5.0],
            [5.0, 7.0],
        ],
        dtype=np.float64,
    ),
    "T2_overlap": np.array(
        [
            [2.0, 1.0],
            [5.0, 1.0],
            [2.0, 4.0],
        ],
        dtype=np.float64,
    ),
    "T3_contained": np.array(
        [
            [0.5, 0.5],
            [1.5, 0.5],
            [0.5, 1.5],
        ],
        dtype=np.float64,
    ),
    "T4_touch_vertex": np.array(
        [
            [4.0, 0.0],
            [6.0, 0.0],
            [4.0, 2.0],
        ],
        dtype=np.float64,
    ),
    "T5_touch_edge": np.array(
        [
            [0.0, 0.0],
            [4.0, 0.0],
            [2.0, -2.0],
        ],
        dtype=np.float64,
    ),
    "T6_crossing": np.array(
        [
            [2.0, -1.0],
            [5.0, 2.0],
            [2.0, 5.0],
        ],
        dtype=np.float64,
    ),
}

triangle_descriptions = {
    "T0_base": "triângulo-base",
    "T1_disjoint": "completamente separado",
    "T2_overlap": "sobreposição parcial",
    "T3_contained": "completamente contido em T0",
    "T4_touch_vertex": "toca T0 em um vértice",
    "T5_touch_edge": "compartilha uma aresta com T0",
    "T6_crossing": "cruza T0",
}

In [70]:
from tca.geometry import triangles_are_disjoint

base_name = "T0_base"
base_triangle = triangles[base_name]

for name, triangle in triangles.items():
    result = triangles_are_disjoint(base_triangle, triangle)
    print(f"{base_name} vs {name:16s} -> {result} ({triangle_descriptions[name]})")

T0_base vs T0_base          -> False (triângulo-base)
T0_base vs T1_disjoint      -> True (completamente separado)
T0_base vs T2_overlap       -> False (sobreposição parcial)
T0_base vs T3_contained     -> False (completamente contido em T0)
T0_base vs T4_touch_vertex  -> False (toca T0 em um vértice)
T0_base vs T5_touch_edge    -> False (compartilha uma aresta com T0)
T0_base vs T6_crossing      -> False (cruza T0)


In [71]:
import pandas as pd

names = list(triangles.keys())

disjoint_matrix = pd.DataFrame(index=names, columns=names, dtype=bool)

for row_name in names:
    for col_name in names:
        disjoint_matrix.loc[row_name, col_name] = triangles_are_disjoint(
            triangles[row_name],
            triangles[col_name],
        )


In [72]:
import numpy as np
import plotly.graph_objects as go


def closed_triangle(triangle):
    triangle = np.asarray(triangle, dtype=np.float64)
    return np.vstack((triangle, triangle[0]))


def add_triangle_trace(fig, triangle, *, name, fillcolor=None, row=None, col=None):
    tri = closed_triangle(triangle)

    trace = go.Scatter(
        x=tri[:, 0],
        y=tri[:, 1],
        mode="lines+markers",
        name=name,
        fill="toself",
        fillcolor=fillcolor,
        line=dict(width=2),
        marker=dict(size=8),
    )

    if row is None or col is None:
        fig.add_trace(trace)
    else:
        fig.add_trace(trace, row=row, col=col)

In [73]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from tca.geometry import triangles_are_disjoint


base_name = "T0_base"
base_triangle = triangles[base_name]

comparison_names = [
    name
    for name in triangles
    if name != base_name
]

comparison_results = {
    name: triangles_are_disjoint(
        base_triangle,
        triangles[name],
    )
    for name in comparison_names
}

fig = make_subplots(
    rows=2,
    cols=3,
    subplot_titles=[
        (
            f"{name}<br>"
            f"{triangle_descriptions[name]}<br>"
            f"disjoint = {comparison_results[name]}"
        )
        for name in comparison_names
    ],
    horizontal_spacing=0.08,
    vertical_spacing=0.15,
)

positions = [
    (1, 1),
    (1, 2),
    (1, 3),
    (2, 1),
    (2, 2),
    (2, 3),
]

for (row, col), name in zip(positions, comparison_names):
    other_triangle = triangles[name]

    add_triangle_trace(
        fig,
        base_triangle,
        name="T0_base",
        fillcolor="rgba(0, 100, 255, 0.20)",
        row=row,
        col=col,
    )

    add_triangle_trace(
        fig,
        other_triangle,
        name=name,
        fillcolor="rgba(255, 100, 0, 0.20)",
        row=row,
        col=col,
    )

    axis_index = (row - 1) * 3 + col
    yaxis = "y" if axis_index == 1 else f"y{axis_index}"

    fig.update_xaxes(
        zeroline=True,
        scaleanchor=yaxis,
        scaleratio=1,
        row=row,
        col=col,
    )

    fig.update_yaxes(
        zeroline=True,
        row=row,
        col=col,
    )

fig.update_layout(
    title="Questão 5 — Casos de disjunção entre triângulos",
    width=1200,
    height=850,
    showlegend=False,
)

fig.show()

## Questão 6

### Critério de convexidade

Para cada três vértices consecutivos $P_i$, $P_{i+1}$ e $P_{i+2}$,
calcula-se o sinal

$$
(P_{i+1}-P_i)\times(P_{i+2}-P_{i+1}).
$$

Um polígono simples é convexo quando todos esses giros possuem o mesmo
sinal.

Nesta implementação, três vértices consecutivos colineares fazem o
algoritmo retornar `False`, pois foi adotada a representação mínima do
polígono, sem vértices intermediários redundantes.

Como cada vértice é analisado uma única vez, a complexidade é

$$
\boxed{O(n)}.
$$

In [74]:
import numpy as np

from tca.geometry import is_convex


convex = np.array(
    [
        [0.0, 0.0],
        [3.0, 0.0],
        [4.0, 2.0],
        [2.0, 4.0],
        [0.0, 2.0],
    ],
    dtype=np.float64,
)

concave = np.array(
    [
        [0.0, 0.0],
        [3.0, 0.0],
        [1.5, 1.0],
        [3.0, 3.0],
        [0.0, 3.0],
    ],
    dtype=np.float64,
)

collinear_vertex = np.array(
    [
        [0.0, 0.0],
        [2.0, 0.0],
        [4.0, 0.0],
        [4.0, 3.0],
        [0.0, 3.0],
    ],
    dtype=np.float64,
)

convex_clockwise = convex[::-1].copy()


print("Convex CCW:", is_convex(convex))
print("Concave:", is_convex(concave))
print("Collinear vertex:", is_convex(collinear_vertex))
print("Convex CW: ", is_convex(convex_clockwise))

Convex CCW: True
Concave: False
Collinear vertex: False
Convex CW:  True


In [75]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from tca.geometry import is_convex


convex_clockwise = convex[::-1].copy()

polygons = {
    "Convex CCW": convex,
    "Convex CW": convex_clockwise,
    "Concave": concave,
    "Collinear vertex": collinear_vertex,
}

fig = make_subplots(
    rows=1,
    cols=4,
    subplot_titles=[
        f"{name}<br>is_convex = {is_convex(polygon)}"
        for name, polygon in polygons.items()
    ],
    horizontal_spacing=0.06,
)

for col, (name, polygon) in enumerate(polygons.items(), start=1):
    closed_polygon = np.vstack((polygon, polygon[0]))

    fig.add_trace(
        go.Scatter(
            x=closed_polygon[:, 0],
            y=closed_polygon[:, 1],
            mode="lines+markers+text",
            text=[
                *(f"P{i + 1}" for i in range(len(polygon))),
                "",
            ],
            textposition="top center",
            fill="toself",
            name=name,
            hovertemplate="x=%{x:.2f}<br>y=%{y:.2f}<extra></extra>",
        ),
        row=1,
        col=col,
    )

    yaxis = "y" if col == 1 else f"y{col}"

    fig.update_xaxes(
        zeroline=True,
        scaleanchor=yaxis,
        scaleratio=1,
        row=1,
        col=col,
    )

    fig.update_yaxes(
        zeroline=True,
        row=1,
        col=col,
    )

fig.update_layout(
    title="Questão 6 — Verificação de convexidade",
    width=1500,
    height=450,
    showlegend=False,
)

fig.show()

## Questão 7

## Questão 7 — Localização a partir da primeira interseção

Considere um polígono simples

$$
P=p_1p_2\ldots p_n
$$

orientado no sentido horário, e um ponto $p_0$ cuja posição em relação ao polígono deseja-se determinar.

A partir de $p_0$, considere uma semirreta e seja $u$ seu primeiro ponto de interseção com a fronteira do polígono, ocorrendo no lado orientado

$$
p_ip_{i+1}.
$$

Como $u$ é a **primeira interseção**, não existe nenhum ponto da fronteira entre $p_0$ e $u$.

Portanto, $p_0$ está na mesma região — interior ou exterior — que o trecho imediatamente anterior à interseção em $u$.

Como o polígono está orientado no sentido horário, seu interior encontra-se à **direita** de cada lado orientado.

Assim, basta verificar de que lado da reta orientada $p_ip_{i+1}$ está o ponto $p_0$.

Definindo

$$
e=p_{i+1}-p_i,
$$

podemos utilizar o sinal do produto vetorial

$$
e\times(p_0-p_i).
$$

Logo,

$$
\boxed{
e\times(p_0-p_i)<0
\quad\Rightarrow\quad
p_0\text{ é interior}
}
$$

e

$$
\boxed{
e\times(p_0-p_i)>0
\quad\Rightarrow\quad
p_0\text{ é exterior}.
}
$$

Se $p_0$ pertencer a algum lado do polígono, então ele está na fronteira.

Se a semirreta não possuir nenhuma interseção com o polígono, então $p_0$ é exterior.

Portanto, conhecendo apenas a primeira interseção $u$ e o lado $p_ip_{i+1}$ em que ela ocorre, a orientação horária do polígono é suficiente para determinar se $p_0$ é interior, exterior ou pertence à fronteira.

In [76]:
import numpy as np


polygon = np.array(
    [
        [0.0, 0.0],
        [-1.0, 4.0],
        [2.0, 1.5],
        [5.0, 3.5],
        [4.0, 0.0],
    ],
    dtype=np.float64,
)

p_inside = np.array([1.0, 1.0], dtype=np.float64)
p_outside_left = np.array([-1.0, 1.5], dtype=np.float64)
p_outside_right = np.array([5.0, 2.0], dtype=np.float64)
p_outside_concavity = np.array([2.0, 2.5], dtype=np.float64)
p_boundary = np.array([2.0, 0.0], dtype=np.float64)
p_outside_tovertex = np.array([-1.0, 0.0], dtype=np.float64)

In [77]:
from tca.geometry import first_horizontal_intersection


points = {
    "p_inside": p_inside,
    "p_outside_left": p_outside_left,
    "p_outside_right": p_outside_right,
    "p_outside_concavity": p_outside_concavity,
    "p_boundary": p_boundary,
    "p_outside_tovertex": p_outside_tovertex,
}

for name, point in points.items():
    intersection = first_horizontal_intersection(point, polygon)

    print(name)

    if intersection is None:
        print("  u = None")
    else:
        print(f"  u = {intersection.point}")
        print(f"  edge_index = {intersection.edge_index}")

    print()

p_inside
  u = [4.285714285714286, 1.0]
  edge_index = 3

p_outside_left
  u = [-0.375, 1.5]
  edge_index = 0

p_outside_right
  u = None

p_outside_concavity
  u = [3.5, 2.5]
  edge_index = 2

p_boundary
  u = [2.0, 0.0]
  edge_index = 4

p_outside_tovertex
  u = [0.0, 0.0]
  edge_index = 0



In [78]:
from tca.geometry import locate_from_first_intersection


LOCATION_LABELS = {
    0: "exterior",
    1: "interior",
    2: "fronteira",
}

for name, point in points.items():
    intersection = first_horizontal_intersection(point, polygon)

    location = locate_from_first_intersection(
        point,
        polygon,
        intersection,
    )

    print(f"{name}: {LOCATION_LABELS[location]}")

p_inside: interior
p_outside_left: exterior
p_outside_right: exterior
p_outside_concavity: exterior
p_boundary: fronteira
p_outside_tovertex: exterior


In [79]:
def visualize_point_in_polygon_results(polygon, points):
    closed_polygon = np.vstack((polygon, polygon[0]))

    fig = go.Figure()

    # Poligono e identificacao dos vertices.
    fig.add_trace(
        go.Scatter(
            x=closed_polygon[:, 0],
            y=closed_polygon[:, 1],
            mode="lines+markers+text",
            text=[
                *(f"P{i + 1}" for i in range(len(polygon))),
                "",
            ],
            textposition="top center",
            fill="toself",
            name="Polígono",
            line=dict(color="black"),
            marker=dict(color="black"),
        )
    )

    ray_end = max(
        polygon[:, 0].max(),
        *(point[0] for point in points.values()),
    ) + 1.0

    for point_name, point in points.items():
        intersection = first_horizontal_intersection(point, polygon)

        location = locate_from_first_intersection(
            point,
            polygon,
            intersection,
        )

        location_label = LOCATION_LABELS[location]
        color = LOCATION_COLORS[location]

        # Ponto p0.
        fig.add_trace(
            go.Scatter(
                x=[point[0]],
                y=[point[1]],
                mode="markers",
                marker=dict(size=12, color=color),
                name=f"{point_name}: {location_label}",
            )
        )

        # Resultado calculado pelo algoritmo C++.
        fig.add_annotation(
            x=point[0],
            y=point[1],
            text=(
                f"{point_name}<br>"
                f"resultado: {location_label}"
            ),
            showarrow=True,
            arrowhead=2,
            ax=0,
            ay=-70,
        )

        if location == 2:
            continue

        if intersection is not None:
            u = np.asarray(intersection.point)
            edge_index = intersection.edge_index

            # Trecho da semirreta entre p0 e a primeira intersecao.
            fig.add_trace(
                go.Scatter(
                    x=[point[0], u[0]],
                    y=[point[1], u[1]],
                    mode="lines",
                    line=dict(color=color, dash="dash"),
                    showlegend=False,
                    hoverinfo="skip",
                )
            )

            # Ponto u.
            fig.add_trace(
                go.Scatter(
                    x=[u[0]],
                    y=[u[1]],
                    mode="markers",
                    marker=dict(
                        size=10,
                        color=color,
                        symbol="x",
                    ),
                    showlegend=False,
                )
            )

            # Identificacao de u e do lado atingido.
            fig.add_annotation(
                x=u[0],
                y=u[1],
                text=(
                    f"u = ({u[0]:.2f}, {u[1]:.2f})"
                    f"<br>P{edge_index + 1} → "
                    f"P{(edge_index + 1) % len(polygon) + 1}"
                ),
                showarrow=True,
                arrowhead=2,
                ax=45,
                ay=45,
            )

        else:
            # Sem intersecao: a semirreta continua para a direita.
            fig.add_trace(
                go.Scatter(
                    x=[point[0], ray_end],
                    y=[point[1], point[1]],
                    mode="lines",
                    line=dict(color=color, dash="dash"),
                    showlegend=False,
                    hoverinfo="skip",
                )
            )

    fig.update_layout(
        title="Questão 7 — Localização pela primeira interseção",
        width=1000,
        height=750,
        xaxis_title="x",
        yaxis_title="y",
        xaxis=dict(
            zeroline=True,
            scaleanchor="y",
            scaleratio=1,
        ),
        yaxis=dict(zeroline=True),
        legend_title="Classificação",
    )

    return fig

In [81]:
LOCATION_LABELS = {
    0: "exterior",
    1: "interior",
    2: "fronteira",
}

LOCATION_COLORS = {
    0: "crimson",
    1: "seagreen",
    2: "darkorange",
}

In [82]:
visualize_point_in_polygon_results(
    polygon,
    points,
).show()